> 在线讲义：[第 28 章 · 负载均衡：专家忙闲不均怎么办（Switch 的 top-1、容量系数、均衡 loss）](https://wikieden.github.io/beyond-transformer/lessons/28-load-balance.html) · 本书《超越 Transformer》

# 第 28 章 · 负载均衡：专家忙闲不均怎么办（Switch 的 top-1、容量系数、均衡 loss）

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/28-load-balance.html) · 仓库里 `beyond-transformer/notes/lessons/28-load-balance.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 容量 | (T/N)·CF | `capacity`、`dropped` | CF 1.0 丢 6 个、1.25 丢 2 个 |
| 第三步 均衡 loss | α·N·Σf·P | `switch_aux_loss(probs)` | 0.0100 vs 0.0280 |
| 第三步 梯度 | α·N·f | `grad_P(f)` | [0.04, 0, 0, 0] |

负载均衡 loss = α · N · Σ f_i · P_i（f 不可导、P 可导），均匀时最小；它的梯度把“太忙”的专家的概率往下压。
完整训练里比较几种均衡方法的负载直方图是里程碑 T-M6。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math

**函数 `switch_aux_loss`**：probs：T × N 的 router softmax。f_i = 被 top-1 分到专家 i 的 token 比例；P_i = 所有 token 给专家 i 的平均概率。

In [ ]:
def switch_aux_loss(probs, alpha=1e-2):
    """probs：T × N 的 router softmax。f_i = 被 top-1 分到专家 i 的 token 比例；P_i = 所有 token 给专家 i 的平均概率。"""
    T, N = probs.shape
    top = probs.argmax(1)
    f = np.bincount(top, minlength=N) / T
    P = probs.mean(0)
    return float(alpha * N * (f * P).sum()), f, P

**函数 `capacity`**：Switch 式 (3)：每个专家最多收 (token 数 / 专家数) × 容量系数 个 token（向下取整）。

In [ ]:
def capacity(tokens, experts, cf):
    """Switch 式 (3)：每个专家最多收 (token 数 / 专家数) × 容量系数 个 token（向下取整）。"""
    return int(math.floor(tokens / experts * cf))

**函数 `dropped`**：按 token 顺序分配，专家满了后来的就被丢掉（不过这一层，只走残差）。

In [ ]:
def dropped(assign, n_exp, cap):
    """按 token 顺序分配，专家满了后来的就被丢掉（不过这一层，只走残差）。"""
    count = np.zeros(n_exp, int)
    drop = 0
    for e in assign:
        if count[e] < cap:
            count[e] += 1
        else:
            drop += 1
    return count, drop

**函数 `grad_P`**：aux loss 对 P_i 的梯度 = α·N·f_i：越忙的专家（f 大），它的概率被压得越狠。

In [ ]:
def grad_P(f, alpha=1e-2):
    """aux loss 对 P_i 的梯度 = α·N·f_i：越忙的专家（f 大），它的概率被压得越狠。"""
    return alpha * len(f) * f

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
N, T = 4, 8
balanced = np.full((T, N), 0.1)
for t in range(T):
    balanced[t, t % N] = 0.7
skewed = np.full((T, N), 0.1)
skewed[:, 0] = 0.7
print("第一步 Switch 的均衡 loss（α = 0.01，4 个专家、8 个 token）")
for name, p in (("均匀分配", balanced), ("全挤到专家 0", skewed)):
    l, f, P = switch_aux_loss(p)
    print(f"  {name}：f = {f.tolist()}，P = {np.round(P, 3).tolist()}，loss = {l:.4f}（均匀时最小 = α = 0.01）")
_, f, _ = switch_aux_loss(skewed)
print(f"  全挤到专家 0 时，loss 对 P 的梯度 = α·N·f = {grad_P(f).tolist()}：只压专家 0 的概率")
print("\n第二步 专家容量与丢 token（32 个 token、4 个专家）")
rng = np.random.default_rng(0)
assign = rng.choice(4, 32, p=[0.4, 0.3, 0.2, 0.1])
for cf in (1.0, 1.25, 2.0):
    cap = capacity(32, 4, cf)
    cnt, d = dropped(assign, 4, cap)
    print(f"  容量系数 {cf}：每个专家最多 {cap} 个；实际 {cnt.tolist()}，丢掉 {d} 个 token")

## 3. 代码里才看得到的

- **“先来先得”决定丢谁**：`dropped` 按 token 顺序分配，所以序列后面的 token 更容易被丢。真实实现有的按 router 分数排优先级，有的随机。
- **f 用的是 top-1**：top-2 时有的实现把两个选择都算进 f，有的只算第一个；同一个公式，不同代码库的数会不一样。

## 4. 自己改一改

- 把 `skewed` 改成“专家 0 和 1 各一半”，loss 是多少？介于均匀和全挤之间吗？
- 把 `dropped` 改成“按专家容量从后往前丢”，丢掉的个数变不变？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 奶茶店富者愈富：偏一点就越偏越狠，浪费参数、拖慢并行、专家会死。
> Switch 只选一个：容量是平均份额乘系数，满了后来的走残差跳过。
> 均衡 loss 阿尔法乘 N 乘 f 乘 P：f 不可导、P 可导；对 P 的梯度是阿尔法 N f，越忙压越狠；均匀时最小。